In [1]:
import sys, os
sys.path.append("../")

import jax
jax.config.update("jax_enable_x64", True)

from qd_solve import *
from qd_solve.eig import *
from qd_solve.operator import *
from qd_solve.split import *
from qd_solve.solver import *
from qd_solve.exp import *

from miscutils.plot import animate

import jax.numpy as jnp
import diffrax

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image

ModuleNotFoundError: No module named 'qd_solve.solver'

In [ ]:
mass = 1.0
hbar = 1.0
potential = lambda x: 0.5 * x ** 2

V = PotentialEnergy(potential)
T = KineticEnergy()

In [ ]:
x0 = -10
xf = 10
num_steps = 1000
x_range = jnp.linspace(x0, xf, num_steps, endpoint=False)
dx = x_range[1] - x_range[0]


y_vals = jnp.ones_like(x_range)
#x_c = 9.0
y_vals = jnp.exp(-0.5 * (x_range - 5) ** 2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.plot(x_range, jax.vmap(potential)(x_range))

plt.savefig("y_vals.png")
plt.show()
plt.close(fig)

In [ ]:
num = 250
mesh = Mesh(x0, xf, num_steps)
y = StateVector.from_values(y_vals, mesh, num)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.plot(x_range, jnp.real(y_vals))
ax.plot(x_range, jnp.imag(y_vals))

plt.savefig("y_vals.png")
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.plot(mesh.x_range, jnp.real(y.values))
ax.plot(mesh.x_range, jnp.imag(y.values))

plt.show()
plt.close(fig)

In [ ]:
Vy = V(y)
Vmesh = jax.vmap(potential)(x_range)
V_direct = Vmesh * y_vals

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.plot(x_range, jnp.real(V_direct))
ax.plot(x_range, jnp.imag(V_direct))

ax.plot(mesh.x_range, jnp.real(Vy.values))
ax.plot(mesh.x_range, jnp.imag(Vy.values))

plt.savefig("Vy_vals.png")
plt.show()
plt.close(fig)

In [ ]:
Ty = T(y)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.plot(mesh.x_range, jnp.real(Ty.values))
ax.plot(mesh.x_range, jnp.imag(Ty.values))

plt.savefig("Ty_vals.png")
plt.show()
plt.close(fig)

In [ ]:
t0 = 0.0
t1 = 2 * jnp.pi
dt = 0.005
t_steps = int((t1 - t0) / (dt))

H = StrangSplitOperator(T, V)
%time ys, t_range = midpoint_solve(H, t0, t1, t_steps, y)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.plot(mesh.x_range, jnp.real(ys[0].values))
ax.plot(mesh.x_range, jnp.imag(ys[0].values))

ax.plot(mesh.x_range, jnp.real(ys[-1].values))
ax.plot(mesh.x_range, jnp.imag(ys[-1].values))

plt.show()
plt.close(fig)

In [ ]:
n = 2

def func(idx, ax, state, args):
    y_vals = ys[n * idx].values
    ax.plot(mesh.x_range, jnp.real(y_vals))
    ax.plot(mesh.x_range, jnp.imag(y_vals))
    ax.plot(mesh.x_range, jnp.abs(y_vals))

    ax.set_xlim([-12, 12])
    ax.set_ylim([-2.0, 2.0])

animate(func, t_steps // n, filename="output.gif")
Image(url="output.gif")  